# 01 · Exploratory Data Analysis & Leakage Purge Audit

**Smart Crop Disease Intelligence System (SKIT/DS/2023-2027/015)**

### Objectives of this notebook:
1. **Dataset Acquisition:** Verify and download PlantVillage (lab), PlantDoc (field proxy), and Crop Recommendation datasets.
2. **Class Imbalance Profiling:** Measure the distribution across all 38 disease categories.
3. **Cross-Split Data Leakage Audit:** Fast parallel MD5 hashing to detect exact duplicate images shared between train, valid, and test.
4. **Leakage Purge Manifest:** Generate `data/leakage_drop_manifest.json` to exclude contaminated images.
5. **Clean Dataset Materialization:** Construct `data/fast/plantvillage/` with verified unleaked samples and package for fast Drive/Kaggle access.
6. **Lab vs. Field Domain Shift Visual Proof:** Side-by-side comparison illustrating why dual-metric reporting is strictly required.


In [ ]:
# --- 1. Universal Environment Setup (Google Colab / Kaggle / Local) ---
import os
import sys
import shutil
import subprocess
from pathlib import Path

# Detect execution platform
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle")

REPO_NAME = "SKIT-DS-2023-2027-015"
REPO_URL = f"https://github.com/shankerjoshi/{REPO_NAME}.git"

if IN_COLAB:
    print("Mounting Google Drive for persistent artifact and model storage...")
    from google.colab import drive
    drive.mount("/content/drive")

    LOCAL_REPO = Path(f"/content/{REPO_NAME}")
    if not LOCAL_REPO.exists():
        print(f"Cloning {REPO_URL} to {LOCAL_REPO}...")
        subprocess.run(["git", "clone", REPO_URL, str(LOCAL_REPO)], check=True)
    os.chdir(str(LOCAL_REPO))
    if str(LOCAL_REPO) not in sys.path:
        sys.path.insert(0, str(LOCAL_REPO))

    # Install dependencies
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime", "shap", "kagglehub", "timm"], check=True)
    print("Colab environment initialized at:", os.getcwd())
    DRIVE_ROOT = Path("/content/drive/MyDrive/PlantDisease_Intelligence")

elif IN_KAGGLE:
    LOCAL_REPO = Path(f"/kaggle/working/{REPO_NAME}")
    if not LOCAL_REPO.exists():
        subprocess.run(["git", "clone", REPO_URL, str(LOCAL_REPO)], check=True)
    os.chdir(str(LOCAL_REPO))
    if str(LOCAL_REPO) not in sys.path:
        sys.path.insert(0, str(LOCAL_REPO))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime", "shap", "kagglehub", "timm"], check=True)
    print("Kaggle environment initialized at:", os.getcwd())
    DRIVE_ROOT = Path("/kaggle/working/PlantDisease_Intelligence")

else:
    if os.path.abspath("..") not in sys.path:
        sys.path.insert(0, os.path.abspath(".."))
    print("Local environment initialized at:", os.getcwd())
    DRIVE_ROOT = Path("runs/drive_sync")

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Master artifact directory: {DRIVE_ROOT}")

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Version : {torch.__version__} | Active Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'})")
if not torch.cuda.is_available():
    print("Warning: No GPU detected. In Colab, go to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU.")


In [ ]:
# --- 2. Dataset Download & Verification (kagglehub / KaggleApi) ---
import os
import shutil
from pathlib import Path

PV_DEST = Path("data/raw/plantvillage")
PD_DEST = Path("data/raw/plantdoc")
CROP_DEST = Path("data/raw/crop")

PV_DEST.mkdir(parents=True, exist_ok=True)
PD_DEST.mkdir(parents=True, exist_ok=True)
CROP_DEST.mkdir(parents=True, exist_ok=True)

# 1. Attach Kaggle Input directly if running on Kaggle
if os.path.exists("/kaggle/input"):
    print("Checking /kaggle/input/ for datasets...")
    for p in Path("/kaggle/input").rglob("*"):
        if p.name.lower() in ["train", "new plant diseases dataset(augmented)"] and p.is_dir():
            target = PV_DEST / p.name
            if not target.exists():
                os.symlink(p, target)
        if "crop" in p.name.lower() and p.suffix == ".csv":
            target = CROP_DEST / p.name
            if not target.exists():
                os.symlink(p, target)
        if "plantdoc" in str(p).lower() and p.is_dir():
            target = PD_DEST / p.name
            if not target.exists():
                os.symlink(p, target)

def has_images(dir_path: Path) -> bool:
    for root, _, files in os.walk(dir_path, followlinks=True):
        if any(f.lower().endswith((".jpg", ".jpeg", ".png")) for f in files):
            return True
    return False

# 2. Download via kagglehub if data not found
if not has_images(PV_DEST):
    print("PlantVillage dataset not found locally. Initiating download via kagglehub...")
    import kagglehub
    pv_path = kagglehub.dataset_download("vipoooool/new-plant-diseases-dataset")
    for item in Path(pv_path).iterdir():
        dest = PV_DEST / item.name
        if not dest.exists():
            if item.is_dir():
                shutil.copytree(item, dest)
            else:
                shutil.copy2(item, dest)

if not has_images(PD_DEST):
    print("PlantDoc dataset not found locally. Initiating download via kagglehub...")
    import kagglehub
    pd_path = kagglehub.dataset_download("nirmalsankalana/plantdoc-dataset")
    for item in Path(pd_path).iterdir():
        dest = PD_DEST / item.name
        if not dest.exists():
            if item.is_dir():
                shutil.copytree(item, dest)
            else:
                shutil.copy2(item, dest)

if not list(CROP_DEST.rglob("*.csv")):
    print("Crop dataset not found locally. Initiating download via kagglehub...")
    import kagglehub
    crop_path = kagglehub.dataset_download("atharvaingle/crop-recommendation-dataset")
    for item in Path(crop_path).rglob("*.csv"):
        shutil.copy2(item, CROP_DEST / item.name)

print("All datasets downloaded and verified successfully.")


In [ ]:
# --- 3. Index Images & Map Class Hierarchy ---
import os
import pandas as pd
from pathlib import Path

base_pv = Path("data/raw/plantvillage")

# Locate candidate directories
train_candidates = list(base_pv.rglob("train"))
valid_candidates = list(base_pv.rglob("valid"))
test_candidates = list(base_pv.rglob("test"))

train_dir = train_candidates[0] if train_candidates else None
valid_dir = valid_candidates[0] if valid_candidates else None
test_dir = test_candidates[0] if test_candidates else None

print(f"Train directory: {train_dir}")
print(f"Valid directory: {valid_dir}")
print(f"Test directory : {test_dir}")

rows = []
for split, root_dir in [("train", train_dir), ("valid", valid_dir)]:
    if root_dir and root_dir.exists():
        for current_dir, _, files in os.walk(root_dir, followlinks=True):
            class_name = os.path.basename(current_dir)
            if class_name.lower() in ["train", "valid"]:
                continue
            for f in files:
                if f.lower().endswith((".jpg", ".jpeg", ".png")):
                    fp = os.path.join(current_dir, f)
                    rows.append({
                        "split": split,
                        "class": class_name,
                        "path": fp,
                        "filename": f,
                        "size_bytes": os.path.getsize(fp)
                    })

# Add test images
if test_dir and test_dir.exists():
    for current_dir, _, files in os.walk(test_dir, followlinks=True):
        for f in files:
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                fp = os.path.join(current_dir, f)
                rows.append({
                    "split": "test",
                    "class": "unlabeled_test",
                    "path": fp,
                    "filename": f,
                    "size_bytes": os.path.getsize(fp)
                })

df_all = pd.DataFrame(rows)
print(f"\nTotal images indexed: {len(df_all):,}")
print(df_all["split"].value_counts())
print(f"Total disease categories: {df_all[df_all.split == 'train']['class'].nunique()}")


In [ ]:
# --- 4. Class Distribution & Imbalance Analysis ---
import matplotlib.pyplot as plt
from pathlib import Path

EDA_DIR = DRIVE_ROOT / "eda"
EDA_DIR.mkdir(parents=True, exist_ok=True)

train_df = df_all[df_all.split == "train"]
class_counts = train_df["class"].value_counts().sort_values(ascending=True)

imbalance_ratio = class_counts.max() / class_counts.min()
print("Class Imbalance Statistics:")
print(f"  Most common class  : {class_counts.index[-1]} ({class_counts.iloc[-1]} images)")
print(f"  Rarest class       : {class_counts.index[0]} ({class_counts.iloc[0]} images)")
print(f"  Imbalance ratio    : {imbalance_ratio:.2f}")

# Plot horizontal bar chart
fig, ax = plt.subplots(figsize=(10, 11))
class_counts.plot(kind="barh", ax=ax, color="#2b5c8f", edgecolor="black", linewidth=0.5)
ax.set_title("PlantVillage Class Distribution (Training Split)", fontsize=13, fontweight="bold")
ax.set_xlabel("Number of Images", fontsize=11)
ax.set_ylabel("Crop & Disease Category", fontsize=11)
plt.tight_layout()

chart_path = EDA_DIR / "results_eda_class_balance.png"
plt.savefig(chart_path, dpi=120)
plt.savefig("results_eda_class_balance.png", dpi=120)
plt.show()
print(f"Saved class balance chart to: {chart_path}")


In [ ]:
# --- 5. Parallel MD5 Hashing & Cross-Split Leakage Audit ---
import hashlib
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
from tqdm.auto import tqdm

print(f"Auditing all {len(df_all):,} images with 8 worker threads...")

def compute_hash_and_dims(item):
    idx, split, label, path = item
    try:
        with Image.open(path) as im:
            dims = im.size
        hasher = hashlib.md5()
        with open(path, "rb") as f:
            while chunk := f.read(65536):
                hasher.update(chunk)
        return (path, split, label, dims, hasher.hexdigest(), None)
    except Exception as e:
        return (path, split, label, None, None, str(e))

items = [(i, r.split, r["class"], r.path) for i, r in df_all.iterrows()]
results = []
with ThreadPoolExecutor(max_workers=8) as executor:
    for res in tqdm(executor.map(compute_hash_and_dims, items), total=len(items)):
        results.append(res)

audit_df = pd.DataFrame(results, columns=["path", "split", "class", "dimensions", "md5", "error"])

# Dimension check
print("\n--- Image Dimensions Summary ---")
print(audit_df["dimensions"].value_counts(dropna=False))

# Build Hash Sets
train_hashes = set(audit_df[audit_df.split == "train"]["md5"])
valid_hashes = set(audit_df[audit_df.split == "valid"]["md5"])
test_hashes = set(audit_df[audit_df.split == "test"]["md5"])

# Check Overlaps (Leakage)
train_val_overlap = train_hashes.intersection(valid_hashes)
train_test_overlap = train_hashes.intersection(test_hashes)
val_test_overlap = valid_hashes.intersection(test_hashes)

print("\n--- Cross-Split Data Leakage Audit ---")
print(f"Train <-> Valid overlap: {len(train_val_overlap)} shared MD5 hashes")
print(f"Train <-> Test overlap : {len(train_test_overlap)} shared MD5 hashes")
print(f"Valid <-> Test overlap : {len(val_test_overlap)} shared MD5 hashes (Contamination in raw test set!)")


In [ ]:
# --- 6. Build & Save Leakage Drop Manifest ---
import json
from collections import defaultdict

hash_map = defaultdict(lambda: {"train": [], "valid": [], "test": []})
for _, row in audit_df.iterrows():
    if row["md5"]:
        hash_map[row["md5"]][row["split"]].append({
            "path": row["path"],
            "filename": os.path.basename(row["path"]),
            "class": row["class"],
            "size_bytes": os.path.getsize(row["path"]) if os.path.exists(row["path"]) else 0
        })

drop_from_train = set()
drop_from_valid = set()
breakdown = {
    "train_leaked_into_test": [],
    "valid_leaked_into_test": [],
    "train_leaked_into_valid": [],
    "train_internal_duplicates": [],
    "valid_internal_duplicates": []
}

for md5, splits in hash_map.items():
    has_test = len(splits["test"]) > 0
    has_valid = len(splits["valid"]) > 0
    has_train = len(splits["train"]) > 0

    # Rule 1: Leaked into Test -> drop from Train and Valid
    if has_test:
        if has_train:
            for item in splits["train"]:
                drop_from_train.add(item["path"])
                breakdown["train_leaked_into_test"].append({"drop_path": item["path"], "md5": md5})
        if has_valid:
            for item in splits["valid"]:
                drop_from_valid.add(item["path"])
                breakdown["valid_leaked_into_test"].append({"drop_path": item["path"], "md5": md5})

    # Rule 2: Leaked between Train and Valid -> drop from Train
    if has_train and has_valid and not has_test:
        for item in splits["train"]:
            drop_from_train.add(item["path"])
            breakdown["train_leaked_into_valid"].append({"drop_path": item["path"], "md5": md5})

    # Rule 3: Internal duplicates within split -> keep first, drop remaining
    if len(splits["train"]) > 1:
        for item in splits["train"][1:]:
            drop_from_train.add(item["path"])
            breakdown["train_internal_duplicates"].append({"drop_path": item["path"], "md5": md5})

    if len(splits["valid"]) > 1:
        for item in splits["valid"][1:]:
            drop_from_valid.add(item["path"])
            breakdown["valid_internal_duplicates"].append({"drop_path": item["path"], "md5": md5})

manifest = {
    "summary": {
        "total_images_analyzed": len(audit_df),
        "total_unique_hashes": len(hash_map),
        "total_to_drop_from_train": len(drop_from_train),
        "total_to_drop_from_valid": len(drop_from_valid),
        "train_leaked_in_test_count": len(breakdown["train_leaked_into_test"]),
        "valid_leaked_in_test_count": len(breakdown["valid_leaked_into_test"]),
        "train_leaked_in_val_count": len(breakdown["train_leaked_into_valid"]),
        "train_internal_duplicate_count": len(breakdown["train_internal_duplicates"]),
        "valid_internal_duplicate_count": len(breakdown["valid_internal_duplicates"]),
    },
    "drop_from_train_paths": sorted(list(drop_from_train)),
    "drop_from_valid_paths": sorted(list(drop_from_valid)),
    "detailed_breakdown": breakdown
}

DATA_MANIFEST_LOCAL = Path("data/leakage_drop_manifest.json")
DATA_MANIFEST_LOCAL.parent.mkdir(parents=True, exist_ok=True)
with open(DATA_MANIFEST_LOCAL, "w") as f:
    json.dump(manifest, f, indent=2)

DATA_DIR_DRIVE = DRIVE_ROOT / "data"
DATA_DIR_DRIVE.mkdir(parents=True, exist_ok=True)
with open(DATA_DIR_DRIVE / "leakage_drop_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("=" * 60)
print("             LEAKAGE & PURGE AUDIT SUMMARY             ")
print("=" * 60)
print(f"Total Images Analyzed:       {len(audit_df):,}")
print(f"Unique MD5 Hashes:           {len(hash_map):,}")
print(f"Images to drop from Train:   {len(drop_from_train):,}")
print(f"Images to drop from Valid:   {len(drop_from_valid):,}")
print(f"Manifest saved to:           {DATA_MANIFEST_LOCAL}")
print(f"Manifest backed up to:       {DATA_DIR_DRIVE / 'leakage_drop_manifest.json'}")
print("=" * 60)


In [ ]:
# --- 7. Materialize Clean Unleaked Dataset in data/fast/ ---
import zipfile

FAST_PV = Path("data/fast/plantvillage")
FAST_TRAIN = FAST_PV / "train"
FAST_VALID = FAST_PV / "valid"

if FAST_TRAIN.exists():
    shutil.rmtree(FAST_TRAIN)
if FAST_VALID.exists():
    shutil.rmtree(FAST_VALID)

FAST_TRAIN.mkdir(parents=True, exist_ok=True)
FAST_VALID.mkdir(parents=True, exist_ok=True)

drop_train_basenames = set(os.path.basename(p) for p in drop_from_train)
drop_valid_basenames = set(os.path.basename(p) for p in drop_from_valid)

def copy_clean_split(src_dir, dest_dir, drop_basenames):
    count = 0
    excluded = 0
    for current_dir, _, files in os.walk(src_dir, followlinks=True):
        class_name = os.path.basename(current_dir)
        if class_name.lower() in ["train", "valid"]:
            continue
        dest_class = dest_dir / class_name
        dest_class.mkdir(parents=True, exist_ok=True)
        for f in files:
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                if f in drop_basenames:
                    excluded += 1
                    continue
                src_path = os.path.join(current_dir, f)
                target_path = dest_class / f
                if not target_path.exists():
                    try:
                        os.symlink(os.path.abspath(src_path), target_path)
                    except Exception:
                        shutil.copy2(src_path, target_path)
                count += 1
    return count, excluded

print("Staging clean training split...")
clean_train, excl_train = copy_clean_split(train_dir, FAST_TRAIN, drop_train_basenames)
print(f"Clean training images: {clean_train:,} (excluded {excl_train})")

print("Staging clean validation split...")
clean_valid, excl_valid = copy_clean_split(valid_dir, FAST_VALID, drop_valid_basenames)
print(f"Clean validation images: {clean_valid:,} (excluded {excl_valid})")

# Package for fast transfer
zip_path = DATA_DIR_DRIVE / "plantvillage_clean_fast.zip"
print(f"\nArchiving clean dataset to Google Drive: {zip_path}...")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(FAST_PV):
        for f in files:
            full_p = os.path.join(root, f)
            rel_p = os.path.relpath(full_p, FAST_PV)
            zf.write(full_p, rel_p)
print(f"Successfully created: {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")


In [ ]:
# --- 8. Lab vs. Field Domain Shift Visual Demonstration ---
import random
from PIL import Image

pv_subset = df_all[df_all.split == "train"].sample(6, random_state=42)["path"].tolist()
pd_files = [str(p) for p in Path("data/raw/plantdoc").rglob("*") if p.suffix.lower() in {".jpg", ".png"}][:6]

fig, axes = plt.subplots(2, 6, figsize=(16, 5.5))
for i, f in enumerate(pv_subset):
    axes[0, i].imshow(Image.open(f))
    axes[0, i].set_title("PlantVillage (Lab)", fontsize=8)
    axes[0, i].axis("off")

for i, f in enumerate(pd_files):
    axes[1, i].imshow(Image.open(f))
    axes[1, i].set_title("PlantDoc (Field)", fontsize=8)
    axes[1, i].axis("off")

plt.suptitle("Lab Setting (Top) vs. Field Proxy (Bottom) — The Architectural Generalization Challenge", fontsize=12, fontweight="bold")
plt.tight_layout()

lab_field_chart = EDA_DIR / "results_eda_lab_vs_field.png"
plt.savefig(lab_field_chart, dpi=120)
plt.savefig("results_eda_lab_vs_field.png", dpi=120)
plt.show()
print(f"Saved lab vs field comparative figure to: {lab_field_chart}")


In [ ]:
# --- 9. Crop Recommendation Dataset Audit ---
crop_files = list(Path("data/raw/crop").rglob("*.csv"))
if crop_files:
    crop_df = pd.read_csv(crop_files[0])
    print(f"Crop dataset shape: {crop_df.shape}")
    print(f"Number of distinct crops: {crop_df['label'].nunique()}")
    print("\nNumerical features summary:")
    display(crop_df.describe().round(2))
    
    crop_summary_path = DATA_DIR_DRIVE / "crop_dataset_summary.csv"
    crop_df.describe().to_csv(crop_summary_path)
    print(f"\nSaved crop summary to {crop_summary_path}")
else:
    print("Crop dataset CSV not found.")


### Summary of Findings for Project Report
- **Class Imbalance:** Significant variation across categories requires class-weighted cross-entropy loss in training.
- **Cross-Split Data Leakage:** Found and purged identical duplicate images across training, validation, and test splits to guarantee unbiased generalization benchmarks.
- **Domain Shift Justification:** PlantVillage's sterile, uniform backgrounds versus PlantDoc's noisy field backgrounds visually and empirically justify reporting two separate accuracy numbers (Lab vs. Field).
